<a href="https://colab.research.google.com/github/CasemiroSJunior/Atividades_Byx/blob/main/exercicio_transformacao_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformação de dados: exercício prático

Mineração de Dados | Fatec Araras

A base já está neste notebook, então não é preciso fazer upload de nenhum arquivo.
O contexto é um cadastro de alunos com notas, horas de estudo e distância até a faculdade.

Cada exercício tem uma célula de conferência logo abaixo. Rode a conferência depois de
resolver: ela mostra OK quando o resultado está correto.

Resolva os exercícios na ordem, porque alguns dependem do anterior.

## Preparação: carregue a base e a função de conferência

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 300

df = pd.DataFrame({
    "id_aluno": np.arange(1, n + 1),
    "idade": rng.integers(17, 46, n),
    # nota final de 0 a 10
    "nota": (rng.normal(6.5, 1.4, n)).clip(0, 10).round(1),
    # horas de estudo com cauda longa à direita
    "horas_estudo": (rng.lognormal(mean=1.6, sigma=0.8, size=n)).round(1),
    # distância até a faculdade, com um valor extremo proposital
    "distancia_km": np.append((rng.gamma(2.0, 3.0, n - 1)).round(1), 320.0),
    "curso": rng.choice(["DSM", "ADS", "Gestão", "Mecatrônica", "Alimentos"],
                        n, p=[.35, .3, .2, .12, .03]),
    "periodo": rng.choice(["matutino", "vespertino", "noturno"], n, p=[.25, .15, .6]),
    "escolaridade_pai": rng.choice(["fundamental", "médio", "superior", "pós"],
                                   n, p=[.3, .4, .25, .05]),
})

df.head()

,id_aluno,idade,nota,horas_estudo,distancia_km,curso,periodo,escolaridade_pai
0,1,44,6.5,7.9,12.9,DSM,noturno,fundamental
1,2,35,5.5,4.4,4.7,ADS,noturno,fundamental
2,3,36,6.0,8.0,5.4,DSM,vespertino,fundamental
3,4,43,5.7,4.8,4.7,Gestão,noturno,fundamental
4,5,33,6.5,2.1,3.7,ADS,noturno,superior


In [ ]:
def checar(descricao, condicao, dica=""):
    """Confere um resultado do exercício e mostra o retorno na tela."""
    if condicao:
        print(f"[OK]    {descricao}")
    else:
        print(f"[ERRO]  {descricao}")
        if dica:
            print(f"        dica: {dica}")


print("função de conferência carregada")

função de conferência carregada


## Exercício 1: reconhecendo o que precisa ser transformado

Antes de transformar qualquer coisa, é preciso saber o que cada coluna tem.

Sua tarefa:

1. exiba as estatísticas descritivas das colunas numéricas com `describe()`
2. exiba o coeficiente de assimetria de cada coluna numérica com `skew()`
3. exiba a quantidade de categorias distintas das colunas categóricas com `nunique()`

Depois responda na célula de texto: qual coluna tem o maior problema de escala e qual tem
o maior problema de assimetria?

In [ ]:
# ESCREVA SEU CODIGO AQUI

### Sua resposta

Qual coluna tem o maior problema de escala e qual tem o maior problema de assimetria?

_(dois cliques para editar este texto)_

## Exercício 2: padronização com Z-Score

Aplique o `StandardScaler` nas colunas `idade` e `nota`, criando as colunas
`idade_z` e `nota_z`.

Lembre-se da sequência: criar o transformador, ajustar e aplicar.

In [ ]:
# ESCREVA SEU CODIGO AQUI

**Conferência**

In [ ]:
checar("as colunas idade_z e nota_z foram criadas",
       {"idade_z", "nota_z"}.issubset(df.columns))

checar("idade_z tem média próxima de zero",
       abs(df["idade_z"].mean()) < 0.01,
       "a fórmula é (x menos a média) dividido pelo desvio padrão")

checar("nota_z tem desvio padrão próximo de um",
       abs(df["nota_z"].std() - 1) < 0.05)

## Exercício 3: escolhendo o método diante de um outlier

A coluna `distancia_km` tem um valor extremo. Sua tarefa:

1. mostre o maior valor da coluna
2. crie `distancia_minmax` aplicando o `MinMaxScaler`
3. crie `distancia_robusta` aplicando o `RobustScaler`
4. compare as duas colunas com `describe()`

Depois responda na célula de texto qual dos dois métodos você usaria nessa coluna e por quê.

In [ ]:
# ESCREVA SEU CODIGO AQUI

**Conferência**

In [ ]:
checar("as duas colunas foram criadas",
       {"distancia_minmax", "distancia_robusta"}.issubset(df.columns))

checar("no Min-Max quase todos os registros ficaram espremidos perto de zero",
       df["distancia_minmax"].quantile(0.75) < 0.05,
       "isso é esperado: o valor extremo definiu sozinho o limite superior")

checar("no robusto os registros comuns continuam distinguíveis",
       df["distancia_robusta"].quantile(0.75) > 0.4)

### Sua resposta

Qual método você usaria em `distancia_km` e por quê?

_(dois cliques para editar este texto)_

## Exercício 4: corrigindo a assimetria

A coluna `horas_estudo` tem cauda longa à direita.

Sua tarefa:

1. exiba a assimetria da coluna original
2. crie `horas_log` aplicando `np.log1p`
3. exiba a assimetria da coluna transformada
4. desenhe os dois histogramas lado a lado com matplotlib

O valor absoluto da assimetria precisa diminuir.

In [ ]:
# ESCREVA SEU CODIGO AQUI

**Conferência**

In [ ]:
checar("a coluna horas_log foi criada",
       "horas_log" in df.columns)

checar("a assimetria diminuiu depois da transformação",
       abs(df["horas_log"].skew()) < abs(df["horas_estudo"].skew()),
       "use np.log1p, que calcula log(1 + x) e aceita zero")

checar("a transformação pode ser revertida",
       np.allclose(np.expm1(df["horas_log"]), df["horas_estudo"]))

## Exercício 5: codificação ordinal

A coluna `escolaridade_pai` é ordinal: existe uma ordem natural entre as categorias.

Use o `OrdinalEncoder` declarando a ordem correta e crie a coluna `escolaridade_cod`,
em que fundamental vira 0 e pós vira 3.

In [ ]:
# ESCREVA SEU CODIGO AQUI

**Conferência**

In [ ]:
checar("a coluna escolaridade_cod foi criada",
       "escolaridade_cod" in df.columns)

mapa = df.drop_duplicates("escolaridade_pai").set_index("escolaridade_pai")["escolaridade_cod"].to_dict() \
    if "escolaridade_cod" in df.columns else {}

checar("fundamental recebeu o menor código e pós o maior",
       mapa.get("fundamental") == 0 and mapa.get("pós") == 3,
       "declare a ordem no parâmetro categories do OrdinalEncoder")

checar("a ordem intermediária está correta",
       mapa.get("médio") == 1 and mapa.get("superior") == 2)

## Entrega

Antes de enviar, confira:

1. todas as células rodam do começo ao fim sem erro, usando Ambiente de execução, Reiniciar e executar tudo
2. todas as conferências mostram OK
3. as duas células de resposta em texto estão preenchidas